At this point we are gonna implement the attention mechanism
1. i need to practice the python around this project, this is the chance to learn the language on the fly.
2. what do we need to implement the attetion in the way how i know about it ??
3. Through an attentions machanism, the text-generating decoder segment of the network is capable of selectively accessing all input tokens, implying that certain input tokens hold more significance than the others in the generation of the specific output token.



. Self attantions in transformer is a technique designed to enhance input representations by enabling each position in the sequence to engage with the and determine the relevance of each other position within the same sequence.

In [2]:
import torch 

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)
# here the each row represents the word, and each column represent the embedding dimention 

In [9]:
input_query = inputs[1]
print(input_query)

tensor([0.5500, 0.8700, 0.6600])


In [10]:
input_1 = inputs[0]
print(input_1)

tensor([0.4300, 0.1500, 0.8900])


In [11]:
result = torch.dot(input_query, input_1)
print(result)

tensor(0.9544)


In [13]:
res = 0 
i= 0 
for idx, element in enumerate(inputs[i]):
    res += inputs[i][idx] * input_query[idx]

print(res)






tensor(0.9544)


In [29]:
query = inputs[1] # the 1 token is the query vector and we are gonna use it to attend to all the other tokens in the input
#i = 0 
attention_scores = torch.empty(inputs.shape[0]) # inputs.shape is (6,3) so inputs.shape[0] is 6. so we will have a tensor with 6 slots, one score per word.
for i, x_i in enumerate(inputs):
     attention_scores[i] = torch.dot(x_i, query)

print(attention_scores)
# here the each x_i is the embedding of the word, and query is the embedding of the word journey.
# we are gonna use the dot product to calculate the attention scores.
# the dot product is a measure of the similarity between two vectors.
# the higher the dot product, the more similar the vectors are.
# so the attention scores will tell us how much each word is attended to by the query word.




tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [25]:
attention_weights_tmp = torch.softmax(attention_scores, dim=0)

In [30]:
print(attention_weights_tmp)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [ ]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(inputs.shape[1])
for i,x_i in enumerate(inputs):
    context_vec_2 += attention_weights_tmp[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


so, the context_vec_2 has go 3 slots in the memory which will be filled by the value of  context_vec_2 += attention_weights_tmp[i]*x_i, the attention weights are from here 
Word      Embedding (x)          Attention weight (α)
Your      [0.43, 0.15, 0.89]     0.1385
journey   [0.55, 0.87, 0.66]     0.2379
starts    [0.57, 0.85, 0.64]     0.2333
with      [0.22, 0.58, 0.33]     0.1240
one       [0.77, 0.25, 0.10]     0.1082
step      [0.05, 0.80, 0.55]     0.1581
                                 ------
                                 sum = 1.0

which are being multiplied to the embedding to get the context vector.

In [39]:
for i, x_i in enumerate(inputs):
    print(i)

0
1
2
3
4
5


Simple self attetion machanism for all the tokens, where you do not have to make the query vector by hardcoding it, the loop or the code will take care of it self. maybe by selecting one input at a time and making it a query vector then attenting it do the other once, and changing the quesry vector on the way also. 

In [ ]:
# query = inputs[1] # the 1 token is the query vector and we are gonna use it to attend to all the other tokens in the input
# #i = 0 
# attention_scores = torch.empty(inputs.shape[0]) # inputs.shape is (6,3) so inputs.shape[0] is 6. so we will have a tensor with 6 slots, one score per word.
# for i, x_i in enumerate(inputs):
#      attention_scores[i] = torch.dot(x_i, query)

# print(attention_scores)
# # here the each x_i is the embedding of the word, and query is the embedding of the word journey.
# # we are gonna use the dot product to calculate the attention scores.
# # the dot product is a measure of the similarity between two vectors.
# # the higher the dot product, the more similar the vectors are.
# # so the attention scores will tell us how much each word is attended to by the query word.


attention_scores = torch.empty(6,6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attention_scores[i,j] = torch.dot(x_i, x_j)

print(attention_scores)


tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([0.0500, 0.8000, 0.5500])
tensor([0.0500, 0.8000, 0.5500])
tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
torch.Size([6, 3])
6
3


In [45]:
# here is the more illustrated version of the self attention machanism. 
words = ["Your", "journey", "starts", "with", "one", "step"]

attention_scores = torch.zeros(6, 6)

for i, x_i in enumerate(inputs):
    print(f"\n===== OUTER LOOP: i = {i}  |  x_i = {words[i]:<8} {x_i} =====")

    for j, x_j in enumerate(inputs):
        products = x_i * x_j                  # element-wise multiply, shape [3]
        score = torch.dot(x_i, x_j)           # sum of those products, a single number
        attention_scores[i, j] = score

        print(f"  inner loop: j = {j}  |  x_j = {words[j]:<8} {x_j}")
        print(f"     ({x_i[0]:.2f}*{x_j[0]:.2f}) + ({x_i[1]:.2f}*{x_j[1]:.2f}) + ({x_i[2]:.2f}*{x_j[2]:.2f})")
        print(f"     = {products[0]:.4f} + {products[1]:.4f} + {products[2]:.4f}"
              f" = {score:.4f}  → stored at attention_scores[{i},{j}]")

    print(f"\nAfter i = {i}, row {i} is filled:\n{attention_scores}")

print("\nFINAL:\n", attention_scores)


===== OUTER LOOP: i = 0  |  x_i = Your     tensor([0.4300, 0.1500, 0.8900]) =====
  inner loop: j = 0  |  x_j = Your     tensor([0.4300, 0.1500, 0.8900])
     (0.43*0.43) + (0.15*0.15) + (0.89*0.89)
     = 0.1849 + 0.0225 + 0.7921 = 0.9995  → stored at attention_scores[0,0]
  inner loop: j = 1  |  x_j = journey  tensor([0.5500, 0.8700, 0.6600])
     (0.43*0.55) + (0.15*0.87) + (0.89*0.66)
     = 0.2365 + 0.1305 + 0.5874 = 0.9544  → stored at attention_scores[0,1]
  inner loop: j = 2  |  x_j = starts   tensor([0.5700, 0.8500, 0.6400])
     (0.43*0.57) + (0.15*0.85) + (0.89*0.64)
     = 0.2451 + 0.1275 + 0.5696 = 0.9422  → stored at attention_scores[0,2]
  inner loop: j = 3  |  x_j = with     tensor([0.2200, 0.5800, 0.3300])
     (0.43*0.22) + (0.15*0.58) + (0.89*0.33)
     = 0.0946 + 0.0870 + 0.2937 = 0.4753  → stored at attention_scores[0,3]
  inner loop: j = 4  |  x_j = one      tensor([0.7700, 0.2500, 0.1000])
     (0.43*0.77) + (0.15*0.25) + (0.89*0.10)
     = 0.3311 + 0.0375 + 0.0

In [47]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
attn_weights = torch.softmax(attn_scores, dim=1)# dim= 1 mean that the softmax will be applied to the rows of the attn_scores matrix which sum up to 1.
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


doing it manually without matrix multiplication would be lot of work, will have to add for loops. This one is just done in 1 line.

In [57]:
contex_vec = attn_weights @ inputs
print(contex_vec)




tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


3.4 Implementing the self-attention with trainable weights.

In [58]:
x_2 = inputs[1]
d_in = inputs.shape[1] # so for the input(6,3) it will be 3.
d_out = 2



In [59]:
torch.manual_seed(123)
w_query = torch.nn.Parameter(torch.rand(d_in,d_out))
w_key = torch.nn.Parameter(torch.rand(d_in,d_out))
w_value = torch.nn.Parameter(torch.rand(d_in,d_out))



In [60]:
query_2 = x_2 @ w_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [62]:
keys = inputs @ w_key
values = inputs @ w_value
keys.shape


torch.Size([6, 2])

In [63]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

This here is just one score so journy compared with journy 

In [65]:
keys_2 = keys[1]
attn_score22 = torch.dot(keys_2, query_2)
attn_score22

tensor(1.8524, grad_fn=<DotBackward0>)

Here we are doing all six scores at once, for the journy quesry vector 
query_2 [2]
keys [6,2] transposed = [2,6]
query_2 @ keys.T = [2] @ [2,6] = [6]


In [67]:
attn_scores = query_2 @ keys.T
attn_scores

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [70]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores/d_k**0.5, dim=-1)
attn_weights_2






tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [71]:
attn_weights_2.sum()

tensor(1., grad_fn=<SumBackward0>)

In [73]:
context_vec_2 = attn_weights_2 @ values
context_vec_2






tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

Implementing the compach attetion class


In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

In [76]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch)
print(batch.shape)





tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
torch.Size([2, 6, 3])


This is the implementation of a compact causal self-attention class

In [80]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # New
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method. 
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
context_length = batch.shape[1]
dropout =0
ca = CausalAttention(d_in, d_out, context_length, dropout)
ca(batch)

tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]],

        [[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)

In [82]:
batch.shape

torch.Size([2, 6, 3])

In [83]:
batch.shape[1]

6

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)